# Projeto: Remoção de ruído periódico e segmentação de letras escondidas

Este notebook foi construído **a partir das ideias das aulas teóricas**:
- imagem digital e pré-processamento;
- filtros no domínio espacial;
- filtros no domínio das frequências;
- segmentação por thresholding.

A estratégia seguida é:

1. **Ler e caracterizar a imagem**  
2. **Aplicar pré-processamento espacial** para reduzir ruído fino  
3. **Analisar o espectro FFT** para encontrar ruído periódico  
4. **Aplicar filtros notch gaussianos simétricos** no domínio da frequência  
5. **Realçar estruturas escuras** no domínio espacial  
6. **Segmentar** com Otsu / threshold local  
7. **Comparar resultados**

O objetivo não é “adivinhar” letras à força, mas montar uma pipeline coerente com a teoria das aulas.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["axes.titlesize"] = 13


## 1. Funções auxiliares

In [ ]:
def resolve_image_path(filename="Letters-noisy.png"):
    candidates = [
        Path(filename),
        Path("/mnt/data") / filename,
        Path.cwd() / filename,
        Path.cwd() / "assets" / filename,
        Path.cwd().parent / "assets" / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Não encontrei '{filename}'. Verifica o caminho.")


def show(img, title="", figsize=(7, 7), cmap="gray"):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis("off")
    plt.show()


def show_pair(a, b, ta, tb, figsize=(14, 6), cmap="gray"):
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    ax[0].imshow(a, cmap=cmap)
    ax[0].set_title(ta)
    ax[0].axis("off")
    ax[1].imshow(b, cmap=cmap)
    ax[1].set_title(tb)
    ax[1].axis("off")
    plt.tight_layout()
    plt.show()


def show_grid(images, titles, ncols=3, figsize=(16, 10), cmap="gray"):
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, ax = plt.subplots(nrows, ncols, figsize=figsize)
    ax = np.array(ax).reshape(nrows, ncols)
    for i in range(nrows * ncols):
        r, c = divmod(i, ncols)
        ax[r, c].axis("off")
        if i < n:
            ax[r, c].imshow(images[i], cmap=cmap)
            ax[r, c].set_title(titles[i])
    plt.tight_layout()
    plt.show()


## 2. Carregar a imagem

In [ ]:
img_path = resolve_image_path("Letters-noisy.png")
img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

if img is None:
    raise RuntimeError("A imagem não foi carregada.")

print("Imagem:", img_path)
print("Shape:", img.shape, "| dtype:", img.dtype)

show(img, "Imagem original")


## 3. Observação inicial

Pela inspeção visual, a imagem contém:
- **ruído periódico** bem marcado;
- uma estrutura central mais escura;
- possíveis letras/traços escondidos.

Isto sugere combinar:
- **filtros espaciais** para redução de ruído fino;
- **filtros no domínio das frequências** para remover o padrão periódico;
- **segmentação** no fim.


In [ ]:
plt.figure(figsize=(8,4))
plt.hist(img.ravel(), bins=256, range=(0,255))
plt.title("Histograma da imagem")
plt.xlabel("Intensidade")
plt.ylabel("Número de píxeis")
plt.show()


## 4. Pré-processamento espacial

In [ ]:
# parâmetros base
GAUSSIAN_KERNEL = 5
MEDIAN_KERNEL = 3

img_blur = cv2.GaussianBlur(img, (GAUSSIAN_KERNEL, GAUSSIAN_KERNEL), 0)
img_med = cv2.medianBlur(img_blur, MEDIAN_KERNEL)

show_grid(
    [img, img_blur, img_med],
    ["Original", "Gaussian blur", "Gaussian + mediana"],
    ncols=3,
    figsize=(15,5)
)


A mediana é útil para reduzir ruído impulsivo/fino, enquanto o suavizamento gaussiano reduz detalhe de alta frequência antes da segmentação.

## 5. FFT e espectro

In [ ]:
def fft_spectrum(img):
    fft = np.fft.fftshift(np.fft.fft2(img.astype(np.float32)))
    magnitude = np.log1p(np.abs(fft))
    return fft, magnitude

fft0, magnitude = fft_spectrum(img_med)

show_pair(img_med, magnitude, "Imagem pré-processada", "Magnitude FFT (log)")


## 6. Deteção automática de picos do ruído periódico

A teoria das aulas de filtragem espectral mostra que ruído periódico surge como **picos localizados no espectro**.  
Como o filtro notch tem de ser **simétrico em relação ao centro**, vamos:
1. ignorar uma região circular central;
2. encontrar máximos locais fortes;
3. construir uma máscara notch gaussiana simétrica.


In [ ]:
PEAK_THRESHOLD_PERCENTILE = 99.9
PEAK_NEIGHBORHOOD = 9
CENTER_EXCLUSION_RADIUS = 16
NOTCH_SIGMA = 6.0

def detect_periodic_peaks(magnitude, percentile=99.9, neighborhood=9, center_radius=12):
    mag = magnitude.copy().astype(np.float32)
    rows, cols = mag.shape
    crow, ccol = rows // 2, cols // 2

    yy, xx = np.ogrid[:rows, :cols]
    center_mask = (yy - crow) ** 2 + (xx - ccol) ** 2 <= center_radius ** 2
    mag[center_mask] = 0

    kernel = np.ones((neighborhood, neighborhood), np.uint8)
    local_max = cv2.dilate(mag, kernel)

    threshold = np.percentile(mag, percentile)
    peak_mask = (mag >= threshold) & (mag == local_max)
    coords = np.argwhere(peak_mask)

    selected = []
    min_dist = max(2, neighborhood // 2)
    for r, c in coords:
        if all((r-r0)**2 + (c-c0)**2 > min_dist**2 for r0, c0 in selected):
            selected.append((int(r), int(c)))

    return selected, peak_mask.astype(np.uint8) * 255


def build_gaussian_notch_mask(shape, peaks, sigma=4.0):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    yy, xx = np.ogrid[:rows, :cols]
    mask = np.ones((rows, cols), np.float32)

    for r, c in peaks:
        rs, cs = 2 * crow - r, 2 * ccol - c

        d2a = (yy - r) ** 2 + (xx - c) ** 2
        d2b = (yy - rs) ** 2 + (xx - cs) ** 2

        notch_a = 1.0 - np.exp(-d2a / (2 * sigma ** 2))
        notch_b = 1.0 - np.exp(-d2b / (2 * sigma ** 2))

        mask *= notch_a * notch_b

    return mask


def apply_frequency_filter(img, mask):
    fft = np.fft.fftshift(np.fft.fft2(img.astype(np.float32)))
    filtered = fft * mask
    rec = np.fft.ifft2(np.fft.ifftshift(filtered))
    rec = np.abs(rec)
    rec = cv2.normalize(rec, None, 0, 255, cv2.NORM_MINMAX)
    return rec.astype(np.uint8)


In [ ]:
peaks, peak_mask = detect_periodic_peaks(
    magnitude,
    percentile=PEAK_THRESHOLD_PERCENTILE,
    neighborhood=PEAK_NEIGHBORHOOD,
    center_radius=CENTER_EXCLUSION_RADIUS
)

mask = build_gaussian_notch_mask(
    img.shape,
    peaks,
    sigma=NOTCH_SIGMA
)

filtered = apply_frequency_filter(img_med, mask)

print("Número de picos detetados:", len(peaks))
show_grid(
    [magnitude, peak_mask, mask, filtered],
    ["Magnitude FFT", "Picos detetados", "Máscara notch", "Após notch filtering"],
    ncols=2,
    figsize=(14,10)
)


## 7. Realce no domínio espacial

Depois de remover parte do padrão periódico, tentamos destacar estruturas escuras com:
- **black-hat morfológico**;
- suavização leve;
- normalização.


In [ ]:
BLACKHAT_KERNEL = 31

kernel_bh = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (BLACKHAT_KERNEL, BLACKHAT_KERNEL))
blackhat = cv2.morphologyEx(filtered, cv2.MORPH_BLACKHAT, kernel_bh)
blackhat = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)

enhanced = cv2.GaussianBlur(blackhat, (3, 3), 0)

show_grid(
    [filtered, blackhat, enhanced],
    ["Filtrada no domínio da frequência", "Black-hat", "Realce final"],
    ncols=3,
    figsize=(16,5)
)


## 8. Segmentação: Otsu e threshold local

In [ ]:
MIN_COMPONENT_AREA = 100

def remove_small_components(binary, min_area=80):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, 8)
    cleaned = np.zeros_like(binary)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            cleaned[labels == i] = 255
    return cleaned

_, otsu = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
adapt = cv2.adaptiveThreshold(
    enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 35, 2
)

otsu_clean = remove_small_components(otsu, MIN_COMPONENT_AREA)
adapt_clean = remove_small_components(adapt, MIN_COMPONENT_AREA)

show_grid(
    [enhanced, otsu, otsu_clean, adapt, adapt_clean],
    ["Realce final", "Otsu", "Otsu limpo", "Threshold local", "Threshold local limpo"],
    ncols=3,
    figsize=(16,10)
)


## 9. Extra: deteção de linhas dominantes (opcional)

Como o ruído ainda tem orientação forte, faz sentido medir isso com bordos + Hough.
Não serve diretamente para ler letras, mas ajuda a perceber se ainda sobraram bandas diagonais.


In [ ]:
edges = cv2.Canny(filtered, 50, 150)
lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=80, minLineLength=40, maxLineGap=8)

line_img = cv2.cvtColor(filtered, cv2.COLOR_GRAY2BGR)
count_lines = 0

if lines is not None:
    for line in lines[:150]:
        x1, y1, x2, y2 = line[0]
        cv2.line(line_img, (x1, y1), (x2, y2), (255, 0, 0), 1)
        count_lines += 1

print("Linhas desenhadas:", count_lines)
show_grid([edges, line_img], ["Canny", "Linhas Hough sobre a imagem filtrada"], ncols=2, figsize=(14,6), cmap=None)


## 10. Zona de experimentação

Aqui podes testar parâmetros sem mexer no resto do notebook.


In [ ]:
# Ajusta estes valores e corre a célula para testar rapidamente
trial_percentile = 99.9
trial_sigma = 6.0
trial_center_radius = 16
trial_neighborhood = 9
trial_blackhat = 31
trial_min_area = 100

peaks_t, peak_mask_t = detect_periodic_peaks(
    magnitude,
    percentile=trial_percentile,
    neighborhood=trial_neighborhood,
    center_radius=trial_center_radius
)

mask_t = build_gaussian_notch_mask(img.shape, peaks_t, sigma=trial_sigma)
filtered_t = apply_frequency_filter(img_med, mask_t)

kernel_t = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (trial_blackhat, trial_blackhat))
blackhat_t = cv2.morphologyEx(filtered_t, cv2.MORPH_BLACKHAT, kernel_t)
blackhat_t = cv2.normalize(blackhat_t, None, 0, 255, cv2.NORM_MINMAX)

_, otsu_t = cv2.threshold(blackhat_t, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
otsu_t = remove_small_components(otsu_t, trial_min_area)

print("Picos:", len(peaks_t))
show_grid(
    [peak_mask_t, mask_t, filtered_t, blackhat_t, otsu_t],
    ["Picos", "Máscara notch", "Filtrada", "Black-hat", "Otsu limpo"],
    ncols=3,
    figsize=(16,10)
)


## 11. Conclusão

Este notebook segue diretamente a lógica das aulas:
- **filtros espaciais** para pré-processamento;
- **análise da DFT** para encontrar periodicidades;
- **notch filtering simétrico** para rejeitar frequências indesejadas;
- **segmentação por thresholding** para isolar regiões.

Na prática, o caso é difícil porque o padrão periódico está muito misturado com a estrutura útil.  
Por isso, o foco do notebook é:
1. montar uma pipeline tecnicamente correta;
2. mostrar resultados intermédios;
3. permitir tuning controlado dos parâmetros.
